In [2]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

**All Required Commands to install MLflow on EC2**

In [3]:
# Test mlflow
import mlflow
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")
with mlflow.start_run():
  mlflow.log_param("param1",15)
  mlflow.log_metric("metric1",0.89)

MlflowException: API request to http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/api/2.0/mlflow/runs/create failed with timeout exception HTTPConnectionPool(host='ec2-52-204-122-132.compute-1.amazonaws.com', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/create (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x79e2ab731be0>, 'Connection to ec2-52-204-122-132.compute-1.amazonaws.com timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120) to a larger value.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/main/data/reddit.csv')
df.head()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df=df[~(df['clean_comment'].str.strip()=='')]

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Ensure that data is downloaded
nltk.download("stopwords")
nltk.donwload("wordnet")

In [ ]:
# Define the preprocessing function
def preprocess_comment(comment):
  comment=comment.lower()
  comment=comment.strip()
  comment=re.sub(r'\n',' ',comment)
  comment=re.sub(r'[^A-Za-z0-9\s!?.,]','',comment)

  # Remove stopwords but retrain important ones for sentiment analysis
  stop_words=set(stopwords.words('english')-{'not','but','however','no','yet'})
  comment=' '.join([word for word in comment.split() if word not in stopwords])

  # Lemmatize the words
  lemmatizer=WordNetLemmatizer()
  comment=' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

  return comment

In [ ]:
# Apply the preprocessing function to the 'clean_comment' column
df['clean_comment']=df['clean_comment'].apply(preprocess_comment)

In [ ]:
df.head()

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StartifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer=CountVectorizer(max_features=10000)

In [ ]:
X=vectorizer.fit_transform(df['clean_comment']).toarray()
y=df['category']

In [ ]:
X

In [ ]:
X.shape